### Definitions

In [133]:
import os
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np

PATH_EXAMPLE_CV = "./data/cv_example.pdf"
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_MODEL_VECTOR_DIMENSIONS = 3072
LLM_TEMPERATURE = 0.5
EMBED_BATCH_SIZE = 100
EMBED_CONCURRENCY = 5
EMBED_MAX_RETRIES = 3

DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings_general"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

In [134]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)", (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_candidate_skills(resume_id: str, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE resume_id = '{resume_id}'")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)    

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """
    Return cosine similarities between a single query vector and a 3D matrix
    of shape (num_indices, size, embedding_dim).
    Zero-padded rows remain zero in the output.
    """
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  


### Get Candidate Info

In [135]:
RESUME_ID = "b6e8165a-b1af-4117-8a29-4a3a5fc95f32"
resume_df = get_resume(RESUME_ID)
candidate_industries = resume_df["industries"][0]

candidate_hard_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_HARD_SKILLS_TABLE)
candidate_soft_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_SOFT_SKILLS_TABLE)

### Get Matching Market Info

In [ ]:
matching_jobs_df = filter_job_postings(candidate_industries)
matching_jobs_ids = matching_jobs_df["id"].to_list()

market_hard_skills_df = get_position_skills(matching_jobs_ids, HARD_SKILLS_TABLE).sort_values("job_id")
market_soft_skills_df = get_position_skills(matching_jobs_ids, SOFT_SKILLS_TABLE).sort_values("job_id")

market_hard_skills_df["index"] = market_hard_skills_df.groupby("job_id").ngroup()
market_soft_skills_df["index"] = market_soft_skills_df.groupby("job_id").ngroup()

### Skills Matrixes

In [ ]:
### STRING, VECTOR and WEIGHT SKills Matrixes
string_hard_skills_df = market_hard_skills_df[["index", "skill_description"]]
string_soft_skills_df = market_soft_skills_df[["index", "skill_description"]]
vector_hard_skills_df = market_hard_skills_df[["index", "embedding"]]
vector_soft_skills_df = market_soft_skills_df[["index", "embedding"]]
weight_hard_skills_df = market_hard_skills_df[["index", "weight"]]
weight_soft_skills_df = market_soft_skills_df[["index", "weight"]]

hard_skills_size = string_hard_skills_df.groupby('index').size().max()
soft_skills_size = string_soft_skills_df.groupby('index').size().max()

string_hard_skills_matrix = np.array([
    np.pad(group['skill_description'].values, (0, hard_skills_size - len(group)), constant_values=0)
    for _, group in string_hard_skills_df.groupby('index')
])
string_soft_skills_matrix = np.array([
    np.pad(group['skill_description'].values, (0, soft_skills_size - len(group)), constant_values=0)
    for _, group in string_soft_skills_df.groupby('index')
])

vector_hard_skills_matrix = np.array([
    np.vstack(list(group['embedding'].values) + [np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)] * (hard_skills_size - len(group)))
    for _, group in vector_hard_skills_df.groupby('index')
], dtype=np.float32)
vector_soft_skills_matrix = np.array([
    np.vstack(list(group['embedding'].values) + [np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)] * (soft_skills_size - len(group)))
    for _, group in vector_soft_skills_df.groupby('index')
], dtype=np.float32)

weight_hard_skills_matrix = np.array([
    np.pad(group['weight'].values, (0, hard_skills_size - len(group)), constant_values=0)
    for _, group in weight_hard_skills_df.groupby('index')
], dtype=np.float32)
weight_soft_skills_matrix = np.array([
    np.pad(group['weight'].values, (0, soft_skills_size - len(group)), constant_values=0)
    for _, group in weight_soft_skills_df.groupby('index')
], dtype=np.float32)

### Find market best matches

For each soft skill

In [207]:
SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.69
SOFT_SKILLS_EMBEDDING_COLUMN_INDEX = 2
SOFT_SKILLS_STRING_COLUMN_INDEX = 4
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
skills_count = candidate_soft_skills_df.shape[0] - 1

for i in range(0,5): #skills_count):
    vector_position = (i, SOFT_SKILLS_STRING_COLUMN_INDEX) 
    string_position = (i, SOFT_SKILLS_EMBEDDING_COLUMN_INDEX)
    weight_position = (i, SOFT_SKILLS_WEIGHT_COLUMN_INDEX)

    weight = candidate_soft_skills_df.iloc[weight_position] 
    skill_embedding = candidate_soft_skills_df.iloc[vector_position] 

    cosine_similarities = cosine_similarities_matrix(skill_embedding, vector_soft_skills_matrix)

    admissible_binary_mask = (cosine_similarities > SOFT_SKILLS_SIMILARITY_THRESHOLD).astype(np.int8)
    admissible_skills_matrix = np.where(admissible_binary_mask == 1, string_soft_skills_matrix, "")

    minimum_weights_matrix = np.where(admissible_binary_mask == 1, weight_soft_skills_matrix, 0)
    ideal_skills_binary_mask = ((weight >= minimum_weights_matrix) & (admissible_binary_mask == 1)).astype(np.int8)
    ideal_skills_matrix = np.where(ideal_skills_binary_mask == 1, string_soft_skills_matrix, "")
    
    unique_admissible_soft_skills = set(admissible_skills_matrix[admissible_skills_matrix != ""].flatten())
    unique_ideal_soft_skills = set(ideal_skills_matrix[ideal_skills_matrix != ""].flatten())
    
    admissible_skill_match_weight = len(unique_admissible_soft_skills)
    ideal_skill_match_weight = len(unique_ideal_soft_skills)

    print(f"*****{candidate_soft_skills_df.iloc[string_position].upper()}*****")
    print(f"There are {len(unique_admissible_soft_skills)} unique pertinent matches: {unique_admissible_soft_skills}")
    print(f"There are {len(unique_ideal_soft_skills)} unique exact matches: {unique_ideal_soft_skills}\n")


*****STAKEHOLDER COMMUNICATION*****
There are 5 unique pertinent matches: {'Stakeholder Collaboration', 'Communication', 'Stakeholder Management', 'Stakeholder Communication', 'Proactive Communication'}
There are 2 unique exact matches: {'Communication', 'Stakeholder Communication'}

*****PROBLEM SOLVING*****
There are 3 unique pertinent matches: {'Problem Solving', 'Problem-Solving', 'Analytical Skills'}
There are 2 unique exact matches: {'Problem Solving', 'Problem-Solving'}

*****TEAMWORK*****
There are 6 unique pertinent matches: {'Team Orientation', 'Engineering Collaboration', 'Teamwork', 'Collaboration', 'Design Collaboration', 'Product Team Collaboration'}
There are 2 unique exact matches: {'Teamwork', 'Collaboration'}

*****AGILE METHODOLOGIES*****
There are 1 unique pertinent matches: {'Agile/Scrum Participation'}
There are 0 unique exact matches: set()

*****DOCUMENTATION*****
There are 0 unique pertinent matches: set()
There are 0 unique exact matches: set()



For each hard skill

In [208]:
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.74
HARD_SKILLS_EMBEDDING_COLUMN_INDEX = 2
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3
skills_count = candidate_hard_skills_df.shape[0] - 1

for i in range(0,5): #skills_count):
    vector_position = (i, HARD_SKILLS_STRING_COLUMN_INDEX) 
    string_position = (i, HARD_SKILLS_EMBEDDING_COLUMN_INDEX)
    weight_position = (i, HARD_SKILLS_WEIGHT_COLUMN_INDEX)

    weight = candidate_hard_skills_df.iloc[weight_position] 
    skill_embedding = candidate_hard_skills_df.iloc[vector_position] 

    cosine_similarities = cosine_similarities_matrix(skill_embedding, vector_hard_skills_matrix)

    admissible_binary_mask = (cosine_similarities > HARD_SKILLS_SIMILARITY_THRESHOLD).astype(np.int8)
    admissible_skills_matrix = np.where(admissible_binary_mask == 1, string_hard_skills_matrix, "")

    minimum_weights_matrix = np.where(admissible_binary_mask == 1, weight_hard_skills_matrix, 0)
    ideal_skills_binary_mask = ((weight >= minimum_weights_matrix) & (admissible_binary_mask == 1)).astype(np.int8)
    ideal_skills_matrix = np.where(ideal_skills_binary_mask == 1, string_hard_skills_matrix, "")
    
    unique_admissible_hard_skills = set(admissible_skills_matrix[admissible_skills_matrix != ""].flatten())
    unique_ideal_hard_skills = set(ideal_skills_matrix[ideal_skills_matrix != ""].flatten())
    
    admissible_skill_match_weight = len(unique_admissible_hard_skills)
    ideal_skill_match_weight = len(unique_ideal_hard_skills)

    print(f"*****{candidate_hard_skills_df.iloc[string_position].upper()}*****")
    print(f"There are {len(unique_admissible_hard_skills)} unique pertinent matches: {unique_admissible_hard_skills}")
    print(f"There are {len(unique_ideal_hard_skills)} unique exact matches: {unique_ideal_hard_skills}\n")


*****DATA PIPELINES*****
There are 4 unique pertinent matches: {'Document Processing Pipelines', 'Data Pipeline Architecture', 'ML Data Pipelines', 'Data Pipelines'}
There are 4 unique exact matches: {'Document Processing Pipelines', 'Data Pipeline Architecture', 'ML Data Pipelines', 'Data Pipelines'}

*****GCP*****
There are 2 unique pertinent matches: {'GCP', 'Google Cloud Platform'}
There are 1 unique exact matches: {'GCP'}

*****AWS*****
There are 2 unique pertinent matches: {'AWS S3', 'AWS'}
There are 2 unique exact matches: {'AWS S3', 'AWS'}

*****MEDALLION MODELS*****
There are 1 unique pertinent matches: {'Medallion Architecture'}
There are 0 unique exact matches: set()

*****DATA MESH*****
There are 0 unique pertinent matches: set()
There are 0 unique exact matches: set()



### Find candidate's missing skills for market 